# 01. Exploratory Data Analysis (EDA) — Turbofan Degradation Telemetry
### Industrial Predictive Maintenance Portfolio Project | NASA C-MAPSS Dataset

---

## 1. Executive Summary & Purpose
This notebook conducts a rigorous, publication-grade Exploratory Data Analysis (EDA) on the NASA **Commercial Modular Aero-Propulsion System Simulation (C-MAPSS)** turbofan engine degradation dataset.

### Key Objectives:
- Ingest raw space-delimited telemetry files using the modular `src.data_loader` package.
- Characterize run-to-failure lifespans across engine units.
- Identify and isolate near-zero variance sensors to prevent overfitting.
- Analyze sensor cross-correlation to detect collinear physical phenomena.
- Characterize operational setting regimes across complex multi-condition subsets (FD002/FD004).

### Theoretical Rationale & Literature Reference:
As established by **Saxena et al. (2008)** in *"Damage Propagation Modeling for Aircraft Engine Run-to-Failure Simulation"* (PHM'08), simulated turbofan engines degrade from an initial healthy state to an unserviceable condition under single or multiple fault modes (High-Pressure Compressor degradation, Fan degradation). Detecting degradation requires isolating informative sensor responses from static atmospheric baselines.

In [ ]:
import sys
from pathlib import Path

# Add project root to path
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_loader import load_dataset, load_rul, COLUMNS, SENSOR_COLUMNS
from src.preprocessor import add_rul_labels, FD001_LOW_VARIANCE_SENSORS

plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
%matplotlib inline

## 2. Telemetry Ingestion & Structural Inspection
We load the FD001 subset (Sea Level operating condition, High-Pressure Compressor failure mode).

In [ ]:
df_fd001 = load_dataset(subset='FD001', split='train')
print(f"FD001 Training Matrix: {df_fd001.shape[0]} cycles across {df_fd001['engine_id'].nunique()} unique engines.")
df_fd001.head()

## 3. Sensor Telemetry Variance Analysis
Sensors operating as constant system references or with zero variance convey no degradation signal and must be pruned prior to modeling. In FD001, sensors `s1, s5, s6, s10, s16, s18, s19` exhibit near-zero variance across all 20,631 cycles.

In [ ]:
variances = df_fd001[SENSOR_COLUMNS].var().sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#e74c3c' if s in FD001_LOW_VARIANCE_SENSORS else '#2980b9' for s in variances.index]
ax.bar(variances.index, variances.values, color=colors, edgecolor='black', linewidth=0.6)
ax.set_yscale('log')
ax.set_xlabel('Sensor Channel')
ax.set_ylabel('Variance (Log Scale)')
ax.set_title('Sensor Telemetry Variance Distribution (Red = Dropped Constant Channels)')
plt.show()

## 4. Engine Lifecycle Distribution
Run-to-failure engines exhibit variable operational longevity before failure, spanning from ~128 cycles to ~362 cycles with a mean near ~206 cycles.

In [ ]:
lifespans = df_fd001.groupby('engine_id')['cycle'].max()

fig, ax = plt.subplots(figsize=(10, 4))
sns.histplot(lifespans, bins=25, kde=True, color='#16a085', ax=ax)
ax.axvline(lifespans.mean(), color='#c0392b', linestyle='--', label=f'Mean: {lifespans.mean():.1f}')
ax.axvline(lifespans.median(), color='#f39c12', linestyle=':', label=f'Median: {lifespans.median():.1f}')
ax.set_xlabel('Total Operating Cycles to Failure')
ax.set_ylabel('Engine Count')
ax.set_title('Distribution of Turbofan Lifecycles (FD001)')
ax.legend()
plt.show()

## 5. Multi-Sensor Degradation Trajectories
Visualizing key thermodynamic and aerodynamic sensors (HPC outlet temperature $s3$, LPT outlet temperature $s4$, HPC outlet static pressure $s11$, fuel flow ratio $s12$) reveals distinct non-linear shifts during the final 50–80 cycles of operation.

In [ ]:
df_labeled = add_rul_labels(df_fd001, clip_rul=False)
sample_engines = [1, 2, 3]
key_sensors = ['s2', 's3', 's4', 's7', 's11', 's12']

fig, axes = plt.subplots(2, 3, figsize=(15, 8), sharex=True)
axes = axes.flatten()

for idx, sensor in enumerate(key_sensors):
    ax = axes[idx]
    for eid in sample_engines:
        subset_df = df_labeled[df_labeled['engine_id'] == eid]
        ax.plot(subset_df['RUL'], subset_df[sensor], label=f'Engine {eid}', alpha=0.8)
    ax.set_title(f'Sensor {sensor} Degradation Profile', fontweight='bold')
    ax.set_ylabel('Sensor Unit')
    ax.invert_xaxis()
    ax.grid(True, alpha=0.3)
    if idx == 0:
        ax.legend()

axes[3].set_xlabel('Remaining Useful Life (RUL -> 0 Failure)')
axes[4].set_xlabel('Remaining Useful Life (RUL -> 0 Failure)')
axes[5].set_xlabel('Remaining Useful Life (RUL -> 0 Failure)')
plt.tight_layout()
plt.show()

## 6. Sensor Cross-Correlation Matrix
Evaluating Pearson correlation among retained sensors highlights strong collinearity between LPC/HPC temperatures ($s2, s3, s4$) and core pressure channels ($s8, s13$). Tree-based gradient boosters are ideally suited to exploit these non-linear interdependencies.

In [ ]:
active_sensors = [s for s in SENSOR_COLUMNS if s not in FD001_LOW_VARIANCE_SENSORS]
corr = df_fd001[active_sensors].corr()

plt.figure(figsize=(11, 9))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='coolwarm', vmin=-1, vmax=1, annot=True, fmt='.2f', square=True, linewidths=0.5)
plt.title('Inter-Sensor Telemetry Correlation Matrix (FD001 Active Sensors)')
plt.show()

## 7. Conclusions & Modeling Guidance
1. **Low-Variance Drop**: 7 sensors (`s1, s5, s6, s10, s16, s18, s19`) must be removed to avoid uninformative noise.
2. **Degradation Horizon**: Healthy engines exhibit stable baseline telemetry until ~100–125 cycles before failure. This motivates **Piecewise Linear RUL Clipping at 125 cycles**.
3. **Temporal Features**: Moving window averages and cumulative wear indicators are needed to filter high-frequency sensor jitter.